# **Building a RAG System with LangChain and FAISS**

Introduction to RAG. RAG comines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data. RAG:

1. Retrieves relevant documents from a knowledge base.
2. Uses these documents as context for the LLM.
3. Generates responses based on both the retrieved context and the model's knowledge.

## FAISS

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantage:
1. Extremely fast similarity search.
2. Memory efficient.
3. Supports GPU acceleration.
4. Can handle millions of vectors.

How it works:

- Indexed vectors for fast nearest neighbor search.
- Returns most similar vectors based on distance metrics.

In [3]:
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from typing import List, Dict, Any, Optional

# Langchain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# Langchain specific imports
from langchain.document_loaders import TextLoader, PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# load environment variables
load_dotenv()

True

In [ ]:
# Data Ingestion and Processing
temp_dir = "/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24"
loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"First document preview: ")
print(documents[0].page_content[:200] + "...")

Loaded 4 documents
First document preview: 

    Data Engineering and ETL Pipelines

    Data Engineering and ETL Pipelines form the backbone of data-driven organizations by enabling ingestion, transformation, storage, and delivery of reliable ...


In [10]:
# Text splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "],
)

document_chunks = text_splitter.split_documents(documents)

print(f"Created {len(document_chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example: ")
print(f"\nContent: {document_chunks[0].page_content[:150]}...")
print(f"\nMetadata: {document_chunks[0].metadata}")

Created 20 chunks from 4 documents

Chunk example: 

Content: Data Engineering and ETL Pipelines

    Data Engineering and ETL Pipelines form the backbone of data-driven organizations by enabling ingestion, trans...

Metadata: {'source': '/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24/doc_3.txt'}


In [13]:
# load embedding models
embeddings = OpenAIEmbeddings(
    base_url=os.getenv("IQ_BASE_URL"),
    model=os.getenv("EMBEDDING_MODEL"),
    dimensions=1536,
)

In [17]:
vectorstore = FAISS.from_documents(
    documents=document_chunks,
    embedding=embeddings,
    
)

In [18]:
vectorstore.save_local("faiss_index")

In [19]:
loaded_vector = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,
)

In [21]:
vectorstore.similarity_search("anything", k=3)

[Document(metadata={'source': '/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24/doc_3.txt'}, page_content='schemas, or wide-table approaches) depends on query patterns and analytical needs. Orchestration tools like Apache Airflow, Dagster, or Prefect manage dependencies, scheduling, and retries while providing observability into pipeline runs. Ensuring data quality requires validation checks, anomaly detection, lineage tracking, and unit tests for transformations. Storage choices—data lakes (object stores like S3 with parquet/ORC formats), data warehouses (Snowflake, BigQuery, Redshift), and'),
 Document(metadata={'source': '/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24/doc_1.txt'}, page_content='testing ensures safe evolution. Finally, capacity planning, autoscaling policies, and chaos engineering exercises help validate that the distributed system can tolerate real-world failures while meeting performance and reliability goals.'),
 Document(metadata={'sourc

In [26]:
# create a retriever handle out of vector store
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {
        "k": 3
    }
)

In [25]:
# create llm model which will be used
llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    base_url=os.getenv("IQ_BASE_URL")
)

test_response = llm.predict("What is LLM?")
print(test_response)

An **LLM** can refer to two different things, depending on the context:  

1. **Large Language Model**  
   - In artificial intelligence, an LLM is a type of machine learning model trained on vast amounts of text data to understand and generate human-like language.  
   - Examples: GPT (like me), Claude, LLaMA, etc.  
   - Uses: answering questions, writing text, summarizing information, translating languages, coding assistance, etc.  
   - Works by predicting the most likely next word or phrase based on patterns learned during training.

2. **Master of Laws (Legum Magister)**  
   - In academia, an LLM is a postgraduate degree in law, typically pursued after obtaining a primary law degree.  
   - Focuses on specialized legal areas like international law, tax law, human rights, etc.  
   - Often used to deepen expertise or qualify for certain roles in legal practice.

---

If you tell me your context — AI or law — I can explain more deeply.  
Do you mean **LLM** as in AI technology or 

In [29]:
simple_prompt = ChatPromptTemplate.from_template(""" Use the following prompt to use the answer.
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support the answer.

Context: {context}

Question: {question}""")

def format_docs(docs: List[Document]) -> str:
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown")
        formatted.append(f"Document {i+1} (source: {source}) : \n{doc.page_content}")
    return "\n\n".join(formatted)



simple_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | llm
    | StrOutputParser()
)

In [30]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x13d105350>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template=" Use the following prompt to use the answer.\nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support the answer.\n\nContext: {context}\n\nQuestion: {question}"))])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x13d6f1c50>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x13dcd5590>, root_client=<openai.OpenAI object at 0x13d686f50>, root_async_client=<openai.AsyncOpenAI object at 0x13d6f0110>, model_name='gpt-5-chat', openai_api_key=SecretS

In [34]:
# Conversational RAG Chain

chat_history = []
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer question."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context} \n\n Question: {input}"),
])

def create_conversational_rag():
    return (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, partial_variables={'chat_history': []}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='You are a helpful AI assistant. Use the provided context to answer question.')), MessagesPlaceholder(variable_name='chat_history', optional=True), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], template='Context: {context} \n\n Question: {input}'))])
| ChatOpenAI(client=<openai.resources.chat.completions.Com

In [41]:
question1 = "What is neural network?"
airesponse = conversational_rag.invoke({
    "input": question1,
    "chat_history": chat_history
})
airesponse

'A neural network is a type of artificial intelligence model composed of layers of interconnected nodes, often called neurons, that can learn complex patterns from data. Each neuron performs a weighted sum of its inputs, followed by a non-linear activation function (such as ReLU, sigmoid, or tanh) to introduce flexibility in modeling. By stacking multiple layers, neural networks can capture increasingly abstract features, enabling them to solve tasks like image recognition, language understanding, and more. They are trained using algorithms like backpropagation combined with gradient-based optimizers, and their architecture can be tailored to specific tasks (e.g., CNNs for images, transformers for sequences).'

In [42]:
chat_history.extend([
    HumanMessage(content=question1),
    AIMessage(content=airesponse)
])

chat_history

[HumanMessage(content='What is neural network?'),
 AIMessage(content='A neural network is a type of artificial intelligence model composed of layers of interconnected nodes, often called neurons, that can learn complex patterns from data. Each neuron performs a weighted sum of its inputs, followed by a non-linear activation function (such as ReLU, sigmoid, or tanh) to introduce flexibility in modeling. By stacking multiple layers, neural networks can capture increasingly abstract features, enabling them to solve tasks like image recognition, language understanding, and more. They are trained using algorithms like backpropagation combined with gradient-based optimizers, and their architecture can be tailored to specific tasks (e.g., CNNs for images, transformers for sequences).')]